In [1]:
import pandas as pd
import os

# suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Regression

In [2]:
# Read the main history file
main_history_path = "Model_Training_History/History_Regression.csv"
main_df = pd.read_csv(main_history_path)
print(f"Main history file loaded with {len(main_df)} rows")
print("Main file columns:", main_df.columns.tolist())

# Store original count for comparison
original_count = len(main_df)

Main history file loaded with 126 rows
Main file columns: ['model_name', 'scaler_name', 'features_list', 'target_column', 'mse', 'mae', 'rmse', 'r2', 'parameters', 'trained_by']


In [3]:
# Define trainer folders and their corresponding names
trainer_configs = [
    {
        'folder': 'Model_Training/Joy_Kumar_Ghosh/Model_Training_History/History_Regression.csv',
        'trainer_name': 'Joy Kumar Ghosh'
    },
    {
        'folder': 'Model_Training/Mezbah_Uddin_Ahmed/Model_Training_History/History_Regression.csv',
        'trainer_name': 'Mezbah Uddin Ahmed'
    },
    {
        'folder': 'Model_Training/Omar_Sadat/Model_Training_History/History_Regression.csv',
        'trainer_name': 'Omar Sadat'
    },
    {
        'folder': 'Model_Training/Sazid_Hasan/Model_Training_History/History_Regression.csv',
        'trainer_name': 'Sazid Hasan'
    }
]

print(f"Will process {len(trainer_configs)} trainer history files:")
for config in trainer_configs:
    if os.path.exists(config['folder']):
        print(f"  ✓ {config['folder']} -> {config['trainer_name']}")
    else:
        print(f"  ✗ {config['folder']} -> {config['trainer_name']} (FILE NOT FOUND)")

Will process 4 trainer history files:
  ✓ Model_Training/Joy_Kumar_Ghosh/Model_Training_History/History_Regression.csv -> Joy Kumar Ghosh
  ✓ Model_Training/Mezbah_Uddin_Ahmed/Model_Training_History/History_Regression.csv -> Mezbah Uddin Ahmed
  ✓ Model_Training/Omar_Sadat/Model_Training_History/History_Regression.csv -> Omar Sadat
  ✓ Model_Training/Sazid_Hasan/Model_Training_History/History_Regression.csv -> Sazid Hasan


In [4]:
# Merge all trainer history files into the main dataframe with duplicate checking
merged_df = main_df.copy()
total_new_rows = 0

for config in trainer_configs:
    file_path = config['folder']
    trainer_name = config['trainer_name']
    
    if not os.path.exists(file_path):
        print(f"\nSkipping {file_path} - File not found")
        continue
    
    try:
        trainer_df = pd.read_csv(file_path)
        print(f"\nProcessing {file_path}:")
        print(f"  - Trainer: {trainer_name}")
        print(f"  - Rows in trainer file: {len(trainer_df)}")
        print(f"  - Columns: {trainer_df.columns.tolist()}")
        
        # Add the 'trained_by' column to trainer dataframe
        trainer_df['trained_by'] = trainer_name
        
        # Before merging, check for duplicates
        before_merge_count = len(merged_df)
        
        # Concatenate the dataframes
        temp_merged = pd.concat([merged_df, trainer_df], ignore_index=True)
        
        # Remove duplicates based on all columns except 'trained_by' 
        # (in case same model was trained by different people)
        columns_for_duplicate_check = [col for col in temp_merged.columns if col != 'trained_by']
        temp_merged_clean = temp_merged.drop_duplicates(subset=columns_for_duplicate_check, keep='first')
        
        new_rows_added = len(temp_merged_clean) - before_merge_count
        duplicates_found = len(temp_merged) - len(temp_merged_clean)
        
        print(f"  - New rows added: {new_rows_added}")
        print(f"  - Duplicates found and removed: {duplicates_found}")
        print(f"  - Total rows after merge: {len(temp_merged_clean)}")
        
        merged_df = temp_merged_clean
        total_new_rows += new_rows_added
        
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")

print(f"\nFinal merged dataframe has {len(merged_df)} rows")
print(f"Total new rows added: {total_new_rows}")
print(f"Original rows: {original_count}")


Processing Model_Training/Joy_Kumar_Ghosh/Model_Training_History/History_Regression.csv:
  - Trainer: Joy Kumar Ghosh
  - Rows in trainer file: 87
  - Columns: ['model_name', 'scaler_name', 'features_list', 'target_column', 'mse', 'mae', 'rmse', 'r2', 'parameters']
  - New rows added: 33
  - Duplicates found and removed: 54
  - Total rows after merge: 159

Processing Model_Training/Mezbah_Uddin_Ahmed/Model_Training_History/History_Regression.csv:
  - Trainer: Mezbah Uddin Ahmed
  - Rows in trainer file: 30
  - Columns: ['model_name', 'scaler_name', 'features_list', 'target_column', 'mse', 'mae', 'rmse', 'r2', 'parameters']
  - New rows added: 1
  - Duplicates found and removed: 29
  - Total rows after merge: 160

Processing Model_Training/Omar_Sadat/Model_Training_History/History_Regression.csv:
  - Trainer: Omar Sadat
  - Rows in trainer file: 42
  - Columns: ['model_name', 'scaler_name', 'features_list', 'target_column', 'mse', 'mae', 'rmse', 'r2', 'parameters']
  - New rows added: 

In [5]:
# Save the merged dataframe back to the original location
if total_new_rows > 0:
    try:
        merged_df.to_csv(main_history_path, index=False)
        print(f"Successfully saved merged data to {main_history_path}")
        print(f"Final file contains {len(merged_df)} rows")
        
        # Display summary of the merged data
        print("\nMerged data summary:")
        print(merged_df.info())
        
        # Show trainer distribution
        if 'trained_by' in merged_df.columns:
            print("\nTrainer distribution:")
            print(merged_df['trained_by'].value_counts())
        
    except Exception as e:
        print(f"Error saving file: {str(e)}")
else:
    print("No new rows to add. File not updated to prevent unnecessary writes.")

Successfully saved merged data to Model_Training_History/History_Regression.csv
Final file contains 160 rows

Merged data summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   model_name     160 non-null    object 
 1   scaler_name    160 non-null    object 
 2   features_list  160 non-null    object 
 3   target_column  160 non-null    object 
 4   mse            160 non-null    float64
 5   mae            160 non-null    float64
 6   rmse           160 non-null    float64
 7   r2             160 non-null    float64
 8   parameters     160 non-null    object 
 9   trained_by     160 non-null    object 
dtypes: float64(4), object(6)
memory usage: 12.6+ KB
None

Trainer distribution:
trained_by
Joy Kumar Ghosh       87
Omar Sadat            42
Mezbah Uddin Ahmed    31
Name: count, dtype: int64


# Classification

In [6]:
# Read the main history file
main_history_path = "Model_Training_History/History_Classification.csv"
main_df = pd.read_csv(main_history_path)
print(f"Main history file loaded with {len(main_df)} rows")
print("Main file columns:", main_df.columns.tolist())

# Store original count for comparison
original_count = len(main_df)

Main history file loaded with 204 rows
Main file columns: ['model_name', 'scaler_name', 'features_list', 'target_column', 'classification_report_dict', 'confusion_matrix_df', 'auc_score_dict', 'parameters', 'trained_by']


In [7]:
# Define trainer folders and their corresponding names for Classification
trainer_configs = [
    {
        'folder': 'Model_Training/Joy_Kumar_Ghosh/Model_Training_History/History_Classification.csv',
        'trainer_name': 'Joy Kumar Ghosh'
    },
    {
        'folder': 'Model_Training/Mezbah_Uddin_Ahmed/Model_Training_History/History_Classification.csv',
        'trainer_name': 'Mezbah Uddin Ahmed'
    },
    {
        'folder': 'Model_Training/Omar_Sadat/Model_Training_History/History_Classification.csv',
        'trainer_name': 'Omar Sadat'
    },
    {
        'folder': 'Model_Training/Sazid_Hasan/Model_Training_History/History_Classification.csv',
        'trainer_name': 'Sazid Hasan'
    }
]

print(f"Will process {len(trainer_configs)} trainer history files:")
for config in trainer_configs:
    if os.path.exists(config['folder']):
        print(f"  ✓ {config['folder']} -> {config['trainer_name']}")
    else:
        print(f"  ✗ {config['folder']} -> {config['trainer_name']} (FILE NOT FOUND)")

Will process 4 trainer history files:
  ✓ Model_Training/Joy_Kumar_Ghosh/Model_Training_History/History_Classification.csv -> Joy Kumar Ghosh
  ✓ Model_Training/Mezbah_Uddin_Ahmed/Model_Training_History/History_Classification.csv -> Mezbah Uddin Ahmed
  ✓ Model_Training/Omar_Sadat/Model_Training_History/History_Classification.csv -> Omar Sadat
  ✓ Model_Training/Sazid_Hasan/Model_Training_History/History_Classification.csv -> Sazid Hasan


In [8]:
# Merge all trainer history files into the main dataframe with duplicate checking
merged_df = main_df.copy()
total_new_rows = 0

for config in trainer_configs:
    file_path = config['folder']
    trainer_name = config['trainer_name']
    
    if not os.path.exists(file_path):
        print(f"\nSkipping {file_path} - File not found")
        continue
    
    try:
        trainer_df = pd.read_csv(file_path)
        print(f"\nProcessing {file_path}:")
        print(f"  - Trainer: {trainer_name}")
        print(f"  - Rows in trainer file: {len(trainer_df)}")
        print(f"  - Columns: {trainer_df.columns.tolist()}")
        
        # Add the 'trained_by' column to trainer dataframe
        trainer_df['trained_by'] = trainer_name
        
        # Before merging, check for duplicates
        before_merge_count = len(merged_df)
        
        # Concatenate the dataframes
        temp_merged = pd.concat([merged_df, trainer_df], ignore_index=True)
        
        # Remove duplicates based on all columns except 'trained_by'
        columns_for_duplicate_check = [col for col in temp_merged.columns if col != 'trained_by']
        temp_merged_clean = temp_merged.drop_duplicates(subset=columns_for_duplicate_check, keep='first')
        
        new_rows_added = len(temp_merged_clean) - before_merge_count
        duplicates_found = len(temp_merged) - len(temp_merged_clean)
        
        print(f"  - New rows added: {new_rows_added}")
        print(f"  - Duplicates found and removed: {duplicates_found}")
        print(f"  - Total rows after merge: {len(temp_merged_clean)}")
        
        merged_df = temp_merged_clean
        total_new_rows += new_rows_added
        
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")

print(f"\nFinal merged dataframe has {len(merged_df)} rows")
print(f"Total new rows added: {total_new_rows}")
print(f"Original rows: {original_count}")


Processing Model_Training/Joy_Kumar_Ghosh/Model_Training_History/History_Classification.csv:
  - Trainer: Joy Kumar Ghosh
  - Rows in trainer file: 159
  - Columns: ['model_name', 'scaler_name', 'features_list', 'target_column', 'classification_report_dict', 'confusion_matrix_df', 'auc_score_dict', 'parameters']
  - New rows added: 0
  - Duplicates found and removed: 159
  - Total rows after merge: 204

Processing Model_Training/Mezbah_Uddin_Ahmed/Model_Training_History/History_Classification.csv:
  - Trainer: Mezbah Uddin Ahmed
  - Rows in trainer file: 0
  - Columns: ['model_name', 'scaler_name', 'features_list', 'target_column', 'classification_report_dict', 'confusion_matrix_df', 'auc_score_dict', 'parameters']
  - New rows added: 0
  - Duplicates found and removed: 0
  - Total rows after merge: 204

Processing Model_Training/Omar_Sadat/Model_Training_History/History_Classification.csv:
  - Trainer: Omar Sadat
  - Rows in trainer file: 36
  - Columns: ['model_name', 'scaler_name',

In [9]:
# Save the merged dataframe back to the original location
if total_new_rows > 0:
    try:
        merged_df.to_csv(main_history_path, index=False)
        print(f"Successfully saved merged data to {main_history_path}")
        print(f"Final file contains {len(merged_df)} rows")
        
        # Display summary of the merged data
        print("\nMerged data summary:")
        print(merged_df.info())
        
        # Show trainer distribution
        if 'trained_by' in merged_df.columns:
            print("\nTrainer distribution:")
            print(merged_df['trained_by'].value_counts())
        
    except Exception as e:
        print(f"Error saving file: {str(e)}")
else:
    print("No new rows to add. File not updated to prevent unnecessary writes.")

No new rows to add. File not updated to prevent unnecessary writes.
